# Recipe 01: The Bell State

**Companion notebook for the Quokka Cookbook**

This notebook prepares the Bell state $|\Phi^+\rangle = (|00\rangle + |11\rangle)/\sqrt{2}$, runs it on a cloud Quokka, and checks it in two measurement bases.

The first run shows the familiar `00`/`11` correlation. The second run changes basis before measurement, which shows the coherence that a classical copied coin would not have. This is not a full Bell test; it is the right first sanity check for the Bell-state circuit.

## Connect to Quokka

The cloud Quokkas are at `quokka1-6.quokkacomputing.com`. We send plain OpenQASM 2.0 by HTTP POST and normalise the response into a `{bitstring: count}` dictionary.

In [ ]:
from collections import Counter
from pathlib import Path

import matplotlib.pyplot as plt
import requests
from requests.packages.urllib3.exceptions import InsecureRequestWarning
requests.packages.urllib3.disable_warnings(InsecureRequestWarning)

# Pick a cloud Quokka (1-6). If one is offline, try another.
QUOKKA = "quokka1"
QUOKKA_URL = f"http://{QUOKKA}.quokkacomputing.com/qsim/qasm"

def _decode_quokka_counts(payload: dict) -> dict:
    """Normalize Quokka responses to a simple {bitstring: count} mapping."""
    if isinstance(payload, dict) and payload and all(isinstance(v, (int, float)) for v in payload.values()):
        return dict(payload)

    if not isinstance(payload, dict):
        raise TypeError(f"Unexpected Quokka payload type: {type(payload)!r}")

    if payload.get("error_code", 0) not in (0, None):
        raise RuntimeError(f"Quokka error {payload.get('error_code')}: {payload.get('error')}")

    result = payload.get("result")
    if not isinstance(result, dict) or "c" not in result:
        raise ValueError(f"Unexpected Quokka result format: {payload}")

    counts = Counter()
    for shot in result["c"]:
        bitstring = ''.join(str(int(bit)) for bit in shot)
        counts[bitstring] += 1
    return dict(counts)

def run_qasm(program: str, shots: int = 1024) -> dict:
    """Send a QASM program to the cloud Quokka and return counts by bitstring."""
    result = requests.post(QUOKKA_URL, json={"script": program, "count": shots}, verify=False)
    result.raise_for_status()
    return _decode_quokka_counts(result.json())

def plot_counts(counts: dict, title: str, labels=("00", "01", "10", "11")):
    values = [counts.get(label, 0) for label in labels]
    plt.figure(figsize=(6, 4))
    plt.bar(labels, values, color="#009688")
    plt.xlabel("Outcome")
    plt.ylabel("Counts")
    plt.title(title)
    plt.tight_layout()
    plt.show()

print(f"Ready to send circuits to {QUOKKA_URL}")

## Run the Bell circuit in the Z basis

The recipe file `bell.qasm` prepares $|\Phi^+\rangle$ and measures immediately in the computational, or Z, basis.

Expected result: only `00` and `11`, up to shot noise.

In [ ]:
bell_qasm = Path("../recipes/01-bell-state/bell.qasm").read_text()
print(bell_qasm)

In [ ]:
z_counts = run_qasm(bell_qasm, shots=1024)
print("Z-basis counts:", z_counts)
plot_counts(z_counts, "Bell state measured in the Z basis")

The Z-basis result confirms correlation: the two measured bits match.

But this histogram alone is not enough to prove entanglement. A classical source could flip one fair coin, copy it, and also produce only `00` and `11`. To see the quantum coherence, we need to ask the state a second question.

## Check the X basis

To measure in the X basis using a standard Z-basis measurement device, apply a Hadamard to each qubit immediately before measuring.

For $|\Phi^+\rangle$, the outcomes should still match. A classical 50/50 mixture of `00` and `11` would not keep this correlation after the basis change.

In [ ]:
x_basis_qasm = """
OPENQASM 2.0;
include "qelib1.inc";

qreg q[2];
creg c[2];

// Prepare |Phi+>
h q[0];
cx q[0], q[1];

// Change from X-basis measurement to ordinary Z-basis readout
h q[0];
h q[1];

measure q[0] -> c[0];
measure q[1] -> c[1];
"""

x_counts = run_qasm(x_basis_qasm, shots=1024)
print("X-basis counts:", x_counts)
plot_counts(x_counts, "Bell state measured in the X basis")

## Distinguish the four Bell states

There are four Bell states. A single measurement basis cannot distinguish all of them: the relative phase is invisible in the Z basis.

The table below is the useful pattern:

| State | Z-basis pattern | X-basis pattern |
|---|---|---|
| $|\Phi^+\rangle$ | same | same |
| $|\Phi^-\rangle$ | same | different |
| $|\Psi^+\rangle$ | different | same |
| $|\Psi^-\rangle$ | different | different |

In [ ]:
bell_preparations = {
    "Phi+": ["h q[0];", "cx q[0], q[1];"],
    "Phi-": ["h q[0];", "cx q[0], q[1];", "z q[0];"],
    "Psi+": ["h q[0];", "cx q[0], q[1];", "x q[1];"],
    "Psi-": ["h q[0];", "cx q[0], q[1];", "z q[0];", "x q[1];"],
}

def make_bell_qasm(prep_lines, basis="Z"):
    lines = [
        "OPENQASM 2.0;",
        "include \"qelib1.inc\";",
        "qreg q[2];",
        "creg c[2];",
        "",
        *prep_lines,
    ]
    if basis == "X":
        lines += ["", "h q[0];", "h q[1];"]
    lines += ["", "measure q[0] -> c[0];", "measure q[1] -> c[1];"]
    return "\n".join(lines)

fig, axes = plt.subplots(2, 4, figsize=(16, 7), sharey=True)

for col, (name, prep) in enumerate(bell_preparations.items()):
    for row, basis in enumerate(("Z", "X")):
        counts = run_qasm(make_bell_qasm(prep, basis=basis), shots=512)
        labels = ["00", "01", "10", "11"]
        values = [counts.get(label, 0) for label in labels]
        ax = axes[row, col]
        ax.bar(labels, values, color="#009688")
        ax.set_title(f"{name}, {basis} basis")
        ax.set_xlabel("Outcome")
        if col == 0:
            ax.set_ylabel("Counts")

plt.suptitle("Bell states in two measurement bases", fontsize=16)
plt.tight_layout()
plt.show()

## What's next?

- Read the [recipe](../recipes/01-bell-state/README.md) for the compact gate-by-gate explanation.
- Try Recipe 02: Teleportation, which uses a Bell pair as a resource.
- When you see Bell states later, remember the lesson from this notebook: one measurement basis can hide the phase information that makes the state quantum.